# gsplat wheel cache (cu130), draft

Builds only the gsplat wheel for this Kaggle image and saves it as `wheels/<key>/gsplat-*.whl` in the output, with a
manifest. **Reads no scene, no checkpoint and no OGC code.** Accelerator: GPU T4 x2 (the key includes the compute
capability). Internet on. After it finishes: save the version, then create a private dataset from its output
(`wheels/` only) and attach it to later sessions; their restore step merges every attached `wheels/`.

Pinned commit: `882fa7f1fc22f17e204a4877be6029232f353e7c` (its `gsplat/` tree `33734f894b0a`, `setup.py` `95f8de51`).

In [ ]:
import glob, hashlib, json, os, subprocess, sys, time
PIN = "882fa7f1fc22f17e204a4877be6029232f353e7c"
SRC, WHEELS, PY = "/tmp/gsplat", "/kaggle/working/wheels", sys.executable
def sh(cmd, cwd=None):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True, cwd=cwd)
if os.path.isdir("/usr/local/cuda/bin"):
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]; os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
sh(f"git clone --recursive --branch bench/gn-vq https://github.com/Daceyyreal/gsplat.git {SRC}")
sh(f"git -C {SRC} checkout -q {PIN} && git -C {SRC} submodule update --init --recursive")
assert subprocess.check_output(["git", "-C", SRC, "rev-parse", "HEAD"], text=True).strip().startswith(PIN)
info = json.loads(subprocess.check_output([PY, "-c", "import json, torch; print(json.dumps({'version': torch.__version__, "
    "'cuda': torch.version.cuda, 'cap': '.'.join(map(str, torch.cuda.get_device_capability(0)))}))"], text=True).strip().splitlines()[-1])
tree = subprocess.check_output(["git", "-C", SRC, "rev-parse", "HEAD:gsplat"], text=True).strip()
setup_rev = subprocess.check_output(["git", "-C", SRC, "rev-parse", "HEAD:setup.py"], text=True).strip()
assert (tree[:12], setup_rev[:8]) == ("33734f894b0a", "95f8de51"), (tree, setup_rev)
key = f"{tree[:12]}-{setup_rev[:8]}-torch{info['version'].replace('+', '_')}-sm{info['cap']}"
os.environ["TORCH_CUDA_ARCH_LIST"], os.environ["MAX_JOBS"] = info["cap"], "2"
with open("/tmp/torch_constraint.txt", "w") as f:
    f.write(f"torch=={info['version']}\n")
out = f"{WHEELS}/{key}"; os.makedirs(out, exist_ok=True)
t0 = time.time()
sh(f"{PY} -m pip wheel -v --no-build-isolation --no-deps -w {out} {SRC} > /kaggle/working/gsplat_build.log 2>&1")
whl = glob.glob(f"{out}/gsplat-*.whl"); assert len(whl) == 1, whl
sha = hashlib.sha256(open(whl[0], "rb").read()).hexdigest()
sh(f"{PY} -m pip install -q -c /tmp/torch_constraint.txt {whl[0]} && {PY} -c 'import gsplat; print(gsplat.__version__)'", cwd="/tmp")
json.dump({"key": key, "wheel": os.path.basename(whl[0]), "sha256": sha, "bytes": os.path.getsize(whl[0]),
           "build_s": time.time() - t0, "commit": PIN, "torch": info}, open(f"{out}/manifest.json", "w"), indent=2)
print(json.dumps(json.load(open(f"{out}/manifest.json")), indent=2))